# 05 — MIDAS vs pyFAI head-to-head

Runs both integrators on the same frame with the same starting geometry,
reports wall-clock time and integrated-intensity agreement. Backs
Paper B (J. Synchrotron Rad.): the 5–13× CPU speedup claim.

Install: `pip install 'midas-integrate[paper]'` — pulls pyFAI + fabio.

In [1]:
import shutil, tempfile, time
from pathlib import Path

import numpy as np
import tifffile

import midas_auto_calibrate as mac
import midas_integrate as mi

try:
    import pyFAI
    try:
        from pyFAI.integrator.azimuthal import AzimuthalIntegrator
    except ImportError:
        from pyFAI.azimuthalIntegrator import AzimuthalIntegrator
    from pyFAI.detectors import Detector
    print('pyFAI', pyFAI.version)
except ImportError:
    AzimuthalIntegrator = None
    print('pyFAI not installed — install via [paper] extra to run the comparison.')

pyFAI 2026.2.1


## Shared setup: stage data, build MIDAS map

In [2]:
workdir = Path(tempfile.mkdtemp(prefix='mi_bench_'))
img_path = workdir / 'CeO2_00001.tif'
shutil.copy(mac.data.CEO2_PILATUS, img_path)
img = tifffile.imread(img_path).astype(np.float64)

# Common geometry + binning for a fair comparison.
cfg = mi.IntegrationConfig(
    lsd=657_436.9, ybc=685.5, zbc=921.0,
    wavelength=0.172973, pixel_size=172.0,
    nr_pixels_y=1475, nr_pixels_z=1679,
    r_min=50, r_max=1000, r_bin_size=0.5,
    eta_min=-180, eta_max=180, eta_bin_size=1.0,
)
artefacts = mi.Mapper(cfg).build(workdir, n_cpus=4)

## MIDAS timing

In [3]:
zarr_zip = workdir / 'CeO2.zarr.zip'
mi.make_zarr_zip(img_path, cfg, zarr_zip)

t0 = time.perf_counter()
result = mi.Integrator(cfg, artefacts).integrate(zarr_zip, n_cpus=4)
midas_seconds = time.perf_counter() - t0
midas_cake = result.load_cake()
print(f'MIDAS: {midas_seconds:.3f} s, cake shape {midas_cake["I"].shape}')

MIDAS: 0.100 s, cake shape (1900, 360)


## pyFAI timing (same geometry, same binning)

In [4]:
if AzimuthalIntegrator is None:
    print('pyFAI not installed; skipping comparison.')
else:
    px_m = cfg.pixel_size * 1e-6
    detector = Detector(pixel1=px_m, pixel2=px_m, max_shape=img.shape)
    ai = AzimuthalIntegrator(
        dist=cfg.lsd * 1e-6,
        poni1=(cfg.ybc + 0.5) * px_m,
        poni2=(cfg.zbc + 0.5) * px_m,
        wavelength=cfg.wavelength * 1e-10,
        detector=detector,
    )
    n_r = len(midas_cake['R'])
    n_eta = len(midas_cake['Eta'])

    t0 = time.perf_counter()
    I_2d, tth, eta = ai.integrate2d(img, n_r, n_eta, unit='2th_deg')
    pyfai_seconds = time.perf_counter() - t0
    print(f'pyFAI:  {pyfai_seconds:.3f} s, cake shape {I_2d.shape}')
    print(f'speedup (pyFAI / MIDAS): {pyfai_seconds / midas_seconds:.2f}×')

pyFAI:  0.775 s, cake shape (360, 1900)
speedup (pyFAI / MIDAS): 7.74×


### Reproducibility notes

- Paper B's headline numbers are averaged over many frames on a beamline
  server; a single-frame notebook call will show run-to-run variance.
- pyFAI's default is single-threaded unless `ai.engine` is reset to
  OpenCL — we benchmark the default user experience.
- Cake comparison: MIDAS indexes as `(nR, nEta)`, pyFAI as `(nEta, nR)`.
  Transpose one to compare directly.